In [ ]:
import { Tuple, RecursiveMap as Map, RecursiveSet as Set, Value } from 'recursive-set';

function set<T extends Value>(...elements: T[]): Set<T> {
    return new Set(...elements);
}

function tpl<T extends Value[]>(...elements: T): Tuple<T> {
    return new Tuple(...elements);
}

# A simple Backtracking Constraint Solver

A `CSP`(short for <em><u>C</u>onstraint <u>S</u>atisfaction <u>P</u>roblem</em>) is defined as a tripel of the form
$$ \langle \texttt{Vars}, \texttt{Values}, \texttt{Constraints} \rangle $$
where 
 * $\texttt{Vars}$ is a list of *variables*,
 * $\texttt{Values}$ is a list of *values* that these variables can take, and 
 * $\texttt{Constraints}$ is a list of formulas describing *conditions* 
   that the variables must satisfy. 
   
   These formulas are given as string.  These strings have to be valid
   TypeScript expressions that evaluate to a Boolean value.
   
To simplify the implementation we assume that the set $\texttt{Values}$ is a set of numbers.

In [ ]:
type Variable = string;
type Formula  = string;
type CSP      = [Variable[], number[], Formula[]];

An `Assignment` maps **some** variables to values.

In [ ]:
type Assignment = Map<Variable, number>;

An `AnnotatedConstraint` is a pair consisting of a `formula` and the set `vars` of all variables occurring in this formula. 

In [ ]:
type AnnotatedConstraint = [Formula, Set<Variable>];

The function `collectVariables(expr)` takes a string `expr` that can be interpreted as a valid expression as its input and collects all variables occurring in `expr`.
It takes care to remove names that correspond to predefined values or functions. To achieve this, it defines a set of `builtIns` containing reserved names (like `Math`, `true`, `false`, `abs`, `min`) that are not treated as variables, ensuring that only true variable names are returned by `collectVariables`.

As I am working here with *regular expressions*, **you do not need to understand what is going on**.

In [ ]:
function collectVariables(expr: Formula): Set<Variable> {
    // The negative lookbehind expression '(?<!\.)' ensures we skip properties like 
    // 'abs' in 'Math.abs', while '\b' ensures we match whole words.
    const identifierRegex = /(?<!\.)\b[a-zA-Z_][a-zA-Z0-9_]*\b/g;
    const variables: Variable[] = [];
    const coreGlobals = new globalThis.Set(['Math', 'true', 'false']);
    let match: RegExpExecArray | null;
    while ((match = identifierRegex.exec(expr)) != null) {
        const candidate = match[0];
        // Because 'abs' is skipped by the regex, we only need to filter 'Math'
        if (!coreGlobals.has(candidate)) {
            variables.push(candidate);
        }
    }
    return set(...variables);
}

In [ ]:
const expr = 'Math.abs(x - y) + Math.abs(z1 - z2)';
collectVariables(expr);

The function `evaluateExpression` takes a string expression and a
variable assignment (context), creates a dynamic function, and evaluates the given expression using the provided values.

In [ ]:
function evaluateExpression(expr: Formula, assign: Assignment, vars: Set<Variable>): boolean {
    const argNames:  Variable[] = [];
    const argValues: number  [] = [];
    for (const v of vars) {
        argNames .push(v);
        argValues.push(assign.get(v));
    }
    const func = new Function(...argNames, `return (${expr});`);
    return func(...argValues);
}

The function `isConsistent` takes four arguments:
- `variable` is a variable.
- `value` is a value that is to be assigned to the variable `variable`.
- `Assignment` is a partial variable assignment that does not assign a value for `variable`
  and that is *consistent* with all constraints in the set `Constraints`.
- `Constraints` is a set of annotated logical formulas.

The function checks whether the assignment
$$ \texttt{assignment} \cup \{\texttt{variable} \mapsto \texttt{value}\}$$
violates any of the formulas in `constraints`. It assumes that
`Assignment` is *consistent*, i.e. all constraints that can already be evaluated with `Assignment`
are evaluated as `true`.

In [ ]:
function isConsistent(
    variable:     Variable,
    value:        number,
    assignment:   Assignment,
    assignedVars: Set<Variable>,
    constraints:  AnnotatedConstraint[]
): boolean {
    const newAssignment = assignment.mutableCopy();
    newAssignment.set(variable, value);
    const newAssignedVars = assignedVars.clone();
    newAssignedVars.add(variable);
    return constraints.every(([formula, vars]) => {
        const canEvaluate = vars.has(variable) && vars.isSubset(newAssignedVars);
        return !canEvaluate || evaluateExpression(formula, newAssignment, vars);
    });
}

Given a **consistent** *partial variable assignment* `assignment` and a constraint satisfaction problem `csp`,
the function `backtrackSearch(assignment, csp)` tries to extend the given assignment recursively and thereby produce a solution of the given CSP.

In [ ]:
function backtrackSearch(
    assignment:   Assignment,
    assignedVars: Set<Variable>,
    variables:    Variable[],
    values:       number[],
    constraints:  AnnotatedConstraint[]
): Assignment | null {   
    if (assignment.size == variables.length) { return assignment; }
    const nextVar = variables.find(v => !assignedVars.has(v));    
    for (const value of values) {
        if (isConsistent(nextVar, value, assignment, 
                         assignedVars, constraints)) 
        {       
            const newAssignment = assignment.mutableCopy();
            newAssignment.set(nextVar, value);    
            const newAssignedVars = assignedVars.clone();
            newAssignedVars.add(nextVar);
            const result = 
                  backtrackSearch(
                      newAssignment, 
                      newAssignedVars, 
                      variables, 
                      values, 
                      constraints);
            if (result != null) { return result; }
        }
    }
    return null;
}

The input to the function `solve` is a *constraint satisfaction problem*, i.e. a *CSP*.
The function `solve` tries to compute a solution of this problem via *backtracking*.
Its main purpose is to transform the given *CSP* into an *annotated CSP* where all the formulas
are *annotated* with their variables. That is, the third component of the *CSP* is now no longer a set of formulas but rather a set of pairs of the form `(f, V)` where `f` is a formula and `V` is the set of variables occurring in this formula. The function then calls the auxiliary function `backtrackSearch` that recursively solves the *annotated CSP*.

In [ ]:
function solve(csp: CSP): Assignment | null {
    const [Vars, Values, Constrs] = csp;
    const annotatedConstraints: AnnotatedConstraint[] = 
          Constrs.map(f => [f, collectVariables(f)]);
    const initialAssignment = new Map<Variable, number>();
    const initialAssignedVars = set<Variable>(); // Empty RecursiveSet
    return backtrackSearch(initialAssignment, initialAssignedVars, 
                           Vars, Values, annotatedConstraints);
}

In [ ]:
// Example Usage to test the new function:
const myCSP: CSP = [
    ['x', 'y', 'z'],   // Variables
    [1, 2, 3, 4, 5],   // Value Domain
    [                  // Constraints
      'x * y == z + 1', 
      'x + y == z'    , 
      'x * y == 2 * z - x * x'
    ]
];

const solution = solve(myCSP);

if (solution) {
    console.log("Solution found!");
    console.log(`x = ${solution.get('x')}`);
    console.log(`y = ${solution.get('y')}`);
    console.log(`z = ${solution.get('z')}`);
} else {
    console.log("No solution exists.");
}